In [19]:

!pip install -q google-genai gradio pytesseract pillow pandas opencv-python-headless
!apt-get update -qq
!apt-get install -y -qq tesseract-ocr

print("All libraries installed successfully!")


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
All libraries installed successfully!


In [28]:

import os
import json
import cv2
import numpy as np
import pandas as pd
import pytesseract
import gradio as gr

from PIL import Image
from google import genai
from google.colab import userdata

# Get API key securely from Colab Secrets
API_KEY = userdata.get("GEMINI_API_KEY")

if not API_KEY:
    raise ValueError(
        "GEMINI_API_KEY not found. Add it in Colab Secrets."
    )

# Initialize Gemini client
client = genai.Client(api_key=API_KEY)

# Use a currently supported Gemini Flash model
MODEL_NAME = "gemini-3.6-flash"

print("Prescription Reader initialized successfully!")
print("Gemini model:", MODEL_NAME)


Prescription Reader initialized successfully!
Gemini model: gemini-3.6-flash


In [32]:

def perform_ocr(image):

    img = np.array(image.convert("RGB"))

    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

    # Resize image for better recognition
    height, width = gray.shape

    gray = cv2.resize(
        gray,
        None,
        fx=2,
        fy=2,
        interpolation=cv2.INTER_CUBIC
    )

    # Reduce image noise
    denoised = cv2.GaussianBlur(
        gray, (3, 3), 0
    )

    # Adaptive thresholding
    processed = cv2.adaptiveThreshold(
        denoised,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        31,
        11
    )

    # OCR extraction
    extracted_text = pytesseract.image_to_string(
        processed,
        config="--oem 3 --psm 6"
    )

    processed_image = Image.fromarray(processed)

    return extracted_text, processed_image


In [33]:

def extract_medicine_details(prescription_text):

    prompt = f"""
You are an assistant that extracts information
from medical prescription text.

Your task is to organize the prescription into
a structured medicine schedule.

IMPORTANT RULES:

1. Extract only information explicitly present.
2. Never invent medicine names or dosages.
3. Preserve the original dosage instructions.
4. Do not provide medical advice.
5. Mark unclear information as "Unclear".
6. Flag uncertain medicine names for verification.
7. Extract duration in days if explicitly mentioned.
8. Do not assume morning, afternoon or night timings.
9. If the prescription says three times daily but
   does not specify exact time slots, do not assign
   morning, afternoon or night automatically.
10. If timing is not specified, write "Unclear".
11. Preserve instructions such as before food,
    after food and once daily in the notes field.

Return ONLY valid JSON in this format:

{{
  "medicines": [
    {{
      "medicine": "Medicine name",
      "dose": "Dose as written",
      "morning": "Instruction or Unclear",
      "afternoon": "Instruction or Unclear",
      "night": "Instruction or Unclear",
      "duration": "Duration in days or Unclear",
      "notes": "Additional instructions or verification required"
    }}
  ]
}}

Prescription text:
{prescription_text}
"""

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config={
            "response_mime_type": "application/json"
        }
    )

    # Convert JSON response into dictionary
    result = json.loads(response.text)

    return result


In [39]:

import tempfile

def process_prescription(image, typed_text):

    typed_text = typed_text or ""

    if image is None and not typed_text.strip():
        raise gr.Error("Please upload an image or enter prescription text.")

    if image is not None and typed_text.strip():
        raise gr.Error("Please use either an image or typed text, not both.")

    processed_image = None

    # STEP 1: Extract text using Pytesseract
    if image is not None:
        try:
            extracted_text, processed_image = perform_ocr(image)

        except Exception as e:
            raise gr.Error(f"OCR failed: {str(e)}")

        if not extracted_text.strip():
            extracted_text = "OCR could not recognize the handwriting."

    else:
        extracted_text = typed_text.strip()

    # STEP 2: Send OCR text AND original image to Gemini
    try:
        if image is not None:
            result = extract_medicine_details(
                extracted_text,
                image
            )
        else:
            result = extract_medicine_details(
                extracted_text
            )

    except Exception as e:
        raise gr.Error(f"Gemini processing failed: {str(e)}")

    # STEP 3: Extract medicine records
    medicines = result.get("medicines", [])

    if not medicines:
        raise gr.Error("No medicine details could be extracted.")

    # STEP 4: Create schedule table
    columns = [
        "medicine",
        "dose",
        "morning",
        "afternoon",
        "night",
        "duration",
        "notes"
    ]

    df = pd.DataFrame(medicines)

    for column in columns:
        if column not in df.columns:
            df[column] = "Unclear"

    df = df[columns].fillna("Unclear")

    # STEP 5: Generate downloadable CSV
    try:
        temp_file = tempfile.NamedTemporaryFile(
            suffix=".csv",
            delete=False
        )

        temp_file.close()

        df.to_csv(temp_file.name, index=False)

    except Exception as e:
        raise gr.Error(f"CSV generation failed: {str(e)}")

    # STEP 6: Return all outputs to Gradio
    return (
        extracted_text,
        processed_image,
        df,
        temp_file.name
    )

print("Prescription processing function updated successfully!")


Prescription processing function updated successfully!


In [40]:

with gr.Blocks(
    theme=gr.themes.Soft(),
    title="Prescription Reader"
) as demo:

    gr.Markdown(
        """
        # 💊 Prescription Reader

        ### AI-Powered Prescription Text Extraction

        Upload a handwritten prescription or enter
        prescription text to generate a structured
        medicine schedule.

        **Powered by Pytesseract OCR and Gemini Flash**
        """
    )

    with gr.Row():

        with gr.Column():

            image_input = gr.Image(
                label="Upload Prescription Image",
                type="pil",
                height=300
            )

            text_input = gr.Textbox(
                label="OR Enter Prescription Text",
                placeholder="Type or paste prescription text here...",
                lines=8
            )

            submit_btn = gr.Button(
                "Extract Medicine Schedule",
                variant="primary"
            )

        with gr.Column():

            processed_output = gr.Image(
                label="Processed Prescription Image",
                height=300
            )

            ocr_output = gr.Textbox(
                label="Extracted OCR / Input Text",
                lines=8,
                interactive=False
            )

    gr.Markdown("## 📋 Extracted Medicine Schedule")

    schedule_output = gr.Dataframe(
        headers=[
            "medicine",
            "dose",
            "morning",
            "afternoon",
            "night",
            "duration",
            "notes"
        ],
        label="Medicine Schedule",
        interactive=False,
        wrap=True
    )

    csv_output = gr.File(
        label="Download Medicine Schedule (CSV)"
    )

    submit_btn.click(
        fn=process_prescription,
        inputs=[
            image_input,
            text_input
        ],
        outputs=[
            ocr_output,
            processed_output,
            schedule_output,
            csv_output
        ]
    )

    gr.Markdown(
        """
        ---
        **Disclaimer:** This application is an automated
        transcription aid, not a medical decision tool.
        Always verify extracted medicine names, doses,
        timings and duration against the original
        prescription with a pharmacist or prescriber.
        """
    )

print("Gradio interface created successfully!")


/tmp/ipykernel_1585/2042165712.py:1: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


Gradio interface created successfully!


In [29]:

test_response = client.models.generate_content(
    model=MODEL_NAME,
    contents="Reply with exactly: Gemini connection successful."
)

print(test_response.text)


Gemini connection successful.


In [ ]:

demo.launch(
    share=True,
    debug=True
)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://c7b01526d1a71ccb6c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/gradio/blocks.py", line 2181, in preprocess_data
    processed_value = await anyio.to_thread.run_sync(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        block.preprocess, inputs_cached, limiter=self.limiter
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/anyio/to_thread.py", line 65, in run_sync
    return await get_async_backend().run_sync_in_worker_thread(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        func, args, abandon_on_cancel=abandon_on_cancel, limiter=limiter
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/anyio/_backends/_asyncio.py", line 2779, in run_sync_in_worker_thread
    return await future
           ^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/anyio/_backends/_asyncio.py", line